In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
character_anime_works = pd.read_csv(r'../data/character_anime_works.csv')
character_anime_works.tail()

,anime_mal_id,character_mal_id,character_name,role
236811,31245,137157,"Shibasaki, Ken",Supporting
236812,36305,136064,"Hamanaka, Midori",Main
236813,36305,133916,"Narumi, Sena",Main
236814,36305,124942,"Hayasaka, Akari",Supporting
236815,34895,155887,"Shiraha, Setsuna",Main


In [4]:
character_anime_works.shape

(236816, 4)

In [5]:
character_anime_works.columns

Index(['anime_mal_id', 'character_mal_id', 'character_name', 'role'], dtype='object')

In [6]:
character_anime_works.dtypes

anime_mal_id         int64
character_mal_id     int64
character_name      object
role                object
dtype: object

In [7]:
character_anime_works['role'].value_counts()

role
Supporting    186285
Main           50531
Name: count, dtype: int64

### Characters' details

In [9]:
characters = pd.read_csv(r'../data/characters.csv')
characters

,character_mal_id,url,name,name_kanji,image,favorites,about
0,280386.0,https://myanimelist.net/character/280386/Envi_...,Envi Mel Champagne,エンヴィ・メル・シャンパーニュ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
1,280354.0,https://myanimelist.net/character/280354/Eleven,Eleven,イレヴン,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
2,280353.0,https://myanimelist.net/character/280353/Stud,Stud,スタッド,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
3,280352.0,https://myanimelist.net/character/280352/Judge,Judge,ジャッジ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
4,280339.0,https://myanimelist.net/character/280339/Eiji_...,Eiji Kurokawa,黒川 英治,https://cdn.myanimelist.net/img/sp/icon/apple-...,0.0,NaN
...,...,...,...,...,...,...,...
209958,282276.0,https://myanimelist.net/character/282276/Farra...,Farrah Van Dorothy,ファラ・ヴァン・ドロシー,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209959,282277.0,https://myanimelist.net/character/282277/Harri...,Harris Mead,ハリス・ミード,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209960,282278.0,https://myanimelist.net/character/282278/Rob,Rob,ロブ,https://cdn.myanimelist.net/images/characters/...,0.0,NaN
209961,282281.0,https://myanimelist.net/character/282281/Grimm,Grimm,グリム,https://cdn.myanimelist.net/images/characters/...,0.0,NaN


In [10]:
characters.shape

(209963, 7)

In [11]:
characters.columns

Index(['character_mal_id', 'url', 'name', 'name_kanji', 'image', 'favorites',
       'about'],
      dtype='object')

In [12]:
characters.dtypes

character_mal_id    float64
url                  object
name                 object
name_kanji           object
image                object
favorites           float64
about                object
dtype: object

Duplicated ID - repeated informations error.

Based on our notes during the course, in this case we should remove them keeping only the first occurrence to avoid changing mean and other statistics

In [13]:
# Counts duplicated ID
duplicate_count = characters['character_mal_id'].duplicated().sum()
print(f"ID duplicati trovati: {duplicate_count}")

# To see duplicated rows
if duplicate_count > 0:
    print(characters[characters['character_mal_id'].duplicated(keep=False)].sort_values(by='character_mal_id'))

ID duplicati trovati: 456
        character_mal_id                                                url  \
208187             546.0       https://myanimelist.net/character/546/Hayate   
208185             546.0       https://myanimelist.net/character/546/Hayate   
208186             546.0       https://myanimelist.net/character/546/Hayate   
204358            4527.0  https://myanimelist.net/character/4527/Narue_N...   
204356            4527.0  https://myanimelist.net/character/4527/Narue_N...   
...                  ...                                                ...   
209850          282115.0    https://myanimelist.net/character/282115/Cavitt   
209859          282131.0  https://myanimelist.net/character/282131/Lieut...   
209858          282131.0  https://myanimelist.net/character/282131/Lieut...   
209733               NaN                                                NaN   
209734               NaN                                                NaN   

                     name

In [ ]:
# Removing duplicates and keeping first record found
#characters = characters.loc[~characters.duplicated(subset=['character_mal_id'])]

# Confirm current status
#print(f"Dataset dopo la rimozione duplicati: {len(characters)} righe")

Column 'favorites' - meaning: describing how many times a user add the character to his/her/its favourites

In [14]:
characters['favorites'].describe()

count    209961.000000
mean         57.616138
std        1197.586488
min           0.000000
25%           0.000000
50%           0.000000
75%           2.000000
max      175632.000000
Name: favorites, dtype: float64

1. **High percentage of zero values**: usual event of large-scale dataset: a  few characters (the famous protagonists) have thousands of likes, while the vast majority of secondary characters have zero or very few of them.
2. **Type anomaly**: The dtype is float64, but the "likes" are necessary integers.

Solutions
- **Memory optimization**: convert column values into int32. Float64 value type takes up a lot of space.

- **Outlier management**: the maximum value of 175.632 is an important signal (the most loved character), not an error. Don't remove but use it to generate a categorization (es. "Top Tier", "Mainstream", "Niche") to encourage analysis for expert users.

- **Cardinality Filter**: If you're only interested in relevant people for your "Data Story", you could create a view of the data frame that excludes those with zero favorites, reducing the cardinality; this allows you to "work with seemingly smaller data" and focus only on the portion of data actually needed for popularity analysis.

In [15]:
# Memory optimization - replacing NaN values with 0 (not in favorites) and convert
characters['favorites'] = characters['favorites'].fillna(0).astype('int32')
print(characters['favorites'].dtypes)

int32


In [ ]:
# Outlier management

In [ ]:
# Cardinality Filter - generating a view which avoid rows with 'favorites' values = 0
characters_fav_view = characters[characters['favorites'] > 0]

# If we don't need original data, reallocate to save memory
# characters = characters[characters['favorites'] > 0]

print(f"View rows (character with fav > 0): {len(characters_fav_view)}")

This convert the highest value to essential knowledge to the audience.

Column 'about' meaning - character's biography

In [16]:
characters['about'].value_counts()

about
No voice actors have been added to this character. Help improve our database by searching for a voice actor, and adding this character to their roles .                                                                                                                                                                                                                                                                                                                                                                                                              199
Appears in episode 1.                                                                                                                                                                                                                                                                                                                                                                                                                                    

**Data Interpretation**: the 199 occurrences of "No voice actors have been added..." and short descriptions like "Appears in episode 1" are signs of poor or missing data, even though they are not formally NaNs.

Thoughts
1. **Replacement**: Is it better to replace Nan with "No info"?
2. **Length Analysis**: Instead of reading the text, create a new column about_length. There's often a correlation between a character's popularity and the length of their description.
3. **Radical Memory Savings**: If you don't plan on doing text analysis, this column takes up a lot of RAM. Out notes suggest that if it's not needed for the current analysis, it's best to drop it or not load it at all to save space.

In [17]:
# Replacement - improves readability
characters['about'] = characters['about'].fillna('No info')

If a journalist reads a profile, he/she'll be albe to understand the information more clearly than an empty space or a technical error.

In [18]:
# Total for each column
print(characters.isna().sum())

# Total of each row which contains at least a NaN value
print(f"Totale righe con almeno un NaN: {characters.isna().any(axis=1).sum()}")

character_mal_id        2
url                     2
name                    2
name_kanji          55480
image                   2
favorites               0
about                   0
dtype: int64
Totale righe con almeno un NaN: 55480


The data extracted shows that 125,436 of 209,963 rows (about 60% of the dataset) have at least one null value.

What does this mean?

- **Massive Loss**: If you used a simple dropna(), you would delete more than half of your database.

- **Nature of the data**: Most of the NaNs are found in _about_ (96,976) and _name_kanji_ (55,480). It's common for many characters to lack a biography or Japanese name in the database.

To do:
- **Avoid Bias**: If you remove a row due to a NaN in an irrelevant column (such as name_kanji if you're doing a statistical analysis of favorites), your analysis will be highly biased, i.e., incorrect.

- **Leverage Pandas' design**: Pandas is designed to skip NaNs during numeric operations, so their presence will not affect the calculation of the favorites average.

- **Critical columns**: Only remove rows if the NaN is present in character_mal_id or name, since without this data the element becomes meaningless.

In [ ]:
# We only remove if essential identity data is missing
#characters = characters.dropna(subset=['character_mal_id', 'name'])

# For the others, we do not delete anything to avoid bias
#print(f"Percentuale di dati preservati: {len(characters)/209963*100:.2f}%")